In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import os

In [2]:
cols_fecha = ['year', 'month', 'day']
cols_elo = ['elo_h', 'elo_a']
# Añadimos las nuevas columnas avanzadas a la lista de variables a escalar
cols_avanzadas = ['rest_days_home', 'rest_days_away', 'avg_pts_scored_home', 
                  'avg_pts_allowed_home', 'avg_pts_scored_away', 'avg_pts_allowed_away',
                  'inactives_home', 'inactives_away'] 
cols_equipos = []

In [3]:
# 1. One-Hot Encoding de fechas y equipos
def preparar_datos_ohe(df, cols_equipos):
    df_copy = df.copy()
    df_copy['game_date'] = pd.to_datetime(df_copy['game_date'])
    df_copy['year'] = df_copy['game_date'].dt.year
    df_copy['month'] = df_copy['game_date'].dt.month
    df_copy['day'] = df_copy['game_date'].dt.day

    df_copy = pd.get_dummies(df_copy, columns=['team_abbreviation_home','team_abbreviation_away'], dtype=float)
    
    if cols_equipos == []:
        cols_equipos = [c for c in df_copy.columns if 'team_abbreviation_home_' in c or 'team_abbreviation_away_' in c]
        
    return df_copy, cols_equipos


In [4]:
# ---  FUNCIÓN PARA PROCESAR JUGADORES INACTIVOS ---
def procesar_lesiones(ruta_inactives, ruta_games):
    print("Procesando tabla de Jugadores Inactivos (Lesiones)...")
    
    # 1. Cargar solo game_id y team_id de los inactivos
    inactives = pd.read_csv(ruta_inactives, usecols=['game_id', 'team_id'])
    
    # 2. Contar cuántos jugadores inactivos hay por partido y por equipo
    inactives_count = inactives.groupby(['game_id', 'team_id']).size().reset_index(name='inactive_count')
    
    # 3. Cargar tabla Game original SOLO para obtener la fecha
    games = pd.read_csv(ruta_games, usecols=['game_id', 'game_date'])
    games['game_date'] = pd.to_datetime(games['game_date'])
    
    # 4. Unir para cambiar el game_id por la fecha
    df_lesiones = pd.merge(inactives_count, games, on='game_id', how='inner')
    
    # Devolvemos Fecha, Equipo y Número de bajas
    return df_lesiones[['game_date', 'team_id', 'inactive_count']]
# --- 2. FUNCIÓN DE FEATURES AVANZADAS (CON LESIONES) ---
def generar_features_avanzadas(df, df_lesiones):
    df_copy = df.copy()
    df_copy['game_date'] = pd.to_datetime(df_copy['game_date'])
    df_copy = df_copy.sort_values('game_date').reset_index(drop=True)

    # Separamos en local y visitante
    home_df = df_copy[['game_date', 'team_id_home', 'pts_home', 'pts_away']].rename(
        columns={'team_id_home': 'team_id', 'pts_home': 'pts_scored', 'pts_away': 'pts_allowed'})
    home_df['is_home'] = 1

    away_df = df_copy[['game_date', 'team_id_away', 'pts_away', 'pts_home']].rename(
        columns={'team_id_away': 'team_id', 'pts_away': 'pts_scored', 'pts_home': 'pts_allowed'})
    away_df['is_home'] = 0

    team_games = pd.concat([home_df, away_df]).sort_values(['team_id', 'game_date']).reset_index(drop=True)

    # --- NUEVO: Añadir conteo de Lesiones del partido actual ---
    team_games = pd.merge(team_games, df_lesiones, on=['game_date', 'team_id'], how='left')
    # Si un equipo no aparece en la tabla ese día, es que tiene 0 lesionados
    team_games['inactive_count'] = team_games['inactive_count'].fillna(0)

    # --- DÍAS DE DESCANSO ---
    team_games['rest_days'] = team_games.groupby('team_id')['game_date'].diff().dt.days
    team_games['rest_days'] = team_games['rest_days'].fillna(14).clip(upper=14)

    # --- MEDIAS MÓVILES PUNTOS (ÚLTIMOS 5 PARTIDOS) ---
    team_games['avg_pts_scored_5'] = team_games.groupby('team_id')['pts_scored'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(100)
    team_games['avg_pts_allowed_5'] = team_games.groupby('team_id')['pts_allowed'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(100)
    
    # Volvemos a separar para juntar con el Dataset principal
    home_features = team_games[team_games['is_home'] == 1][['game_date', 'team_id', 'rest_days', 'avg_pts_scored_5', 'avg_pts_allowed_5', 'inactive_count']]
    home_features.columns = ['game_date', 'team_id_home', 'rest_days_home', 'avg_pts_scored_home', 'avg_pts_allowed_home', 'inactives_home']

    away_features = team_games[team_games['is_home'] == 0][['game_date', 'team_id', 'rest_days', 'avg_pts_scored_5', 'avg_pts_allowed_5', 'inactive_count']]
    away_features.columns = ['game_date', 'team_id_away', 'rest_days_away', 'avg_pts_scored_away', 'avg_pts_allowed_away', 'inactives_away']

    # Fusionamos
    df_copy = pd.merge(df_copy, home_features, on=['game_date', 'team_id_home'], how='left')
    df_copy = pd.merge(df_copy, away_features, on=['game_date', 'team_id_away'], how='left')
    df_copy = df_copy.drop_duplicates(subset=['team_id_home', 'team_id_away', 'game_date'])

    return df_copy

In [5]:
# 3. Escalado de datos
def escalar_datos(df, df_test, cols_no_escalables, cols_escalables):
    scaler = StandardScaler()
    scaler.fit(df[cols_escalables])

    df_escalado = scaler.transform(df[cols_escalables])
    df_test_escalado = scaler.transform(df_test[cols_escalables])
    
    data_entrada = np.hstack([np.array(df[cols_no_escalables]), df_escalado])
    data_entrada_test = np.hstack([np.array(df_test[cols_no_escalables]), df_test_escalado])
    
    return data_entrada, data_entrada_test

def preparar_datos_salida(df):
    return np.column_stack((df['pts_home'].values, df['pts_away'].values))


In [6]:
# ----------------- PROCESAMIENTO DE DATOS -----------------
print("Cargando datos principales...")
df_partidos_elo1 = pd.read_csv('csv_red/partidos_elo1.csv')

RUTA_INACTIVES = "../../data/inputs/csv/inactive_players.csv" 
RUTA_GAME = "../../data/inputs/csv/game.csv" 

# Procesamos las lesiones
df_lesiones = procesar_lesiones(RUTA_INACTIVES, RUTA_GAME)

# Generamos las variables
df_partidos_elo1 = generar_features_avanzadas(df_partidos_elo1, df_lesiones)
df_partidos_elo1, cols_equipos = preparar_datos_ohe(df_partidos_elo1, cols_equipos)

partidos_elo1 = df_partidos_elo1[df_partidos_elo1['season_id'].astype(str).str[-4:].astype(int) <= 2017] 
partidos_elo1_test = df_partidos_elo1[df_partidos_elo1['season_id'].astype(str).str[-4:].astype(int) > 2017] 

todas_cols_escalables = cols_fecha + cols_elo + cols_avanzadas
data_entrada_elo1, data_entrada_elo1_test = escalar_datos(partidos_elo1, partidos_elo1_test, cols_equipos, todas_cols_escalables)
data_salida_elo1, data_salida_elo1_test = preparar_datos_salida(partidos_elo1), preparar_datos_salida(partidos_elo1_test)


Cargando datos principales...
Procesando tabla de Jugadores Inactivos (Lesiones)...


In [7]:
# ----------------- MODELO DE RED NEURONAL -----------------
def crear_modelo_v1_2(n_input):
    modelo = tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation='relu', input_shape=[n_input]),
        tf.keras.layers.Dropout(0.3), 
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2), 
        tf.keras.layers.Dense(2, activation='linear') 
    ])
    
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
        loss=tf.keras.losses.Huber(delta=1.5), 
        metrics=['mean_absolute_error']
    )
    return modelo

def evaluar_precision(modelo, entrada, salida, nombre_modelo):
    predicciones = modelo.predict(entrada, verbose=0)
    mae_home = mean_absolute_error(salida[:, 0], predicciones[:, 0])
    mae_away = mean_absolute_error(salida[:, 1], predicciones[:, 1])
    mae_total = mean_absolute_error(salida, predicciones)
    
    ganador_pred = (predicciones[:, 0] > predicciones[:, 1]).astype(int)
    ganador_real = (salida[:, 0] > salida[:, 1]).astype(int)
    precision = np.mean(ganador_pred == ganador_real) * 100

    print(f"\n--- Resultados {nombre_modelo} ---")
    print(f"Error Promedio Puntos (MAE Total): {mae_total:.2f}")
    print(f"  -> Error Medio Local: {mae_home:.2f}")
    print(f"  -> Error Medio Visitante: {mae_away:.2f}")
    print(f"Precisión Ganador (deducida): {precision:.2f}%")

    return precision, mae_total


In [8]:
# ----------------- ENTRENAMIENTO Y PANTALLA -----------------
modelo_elo1_v1_2 = crear_modelo_v1_2(data_entrada_elo1.shape[1])
print("Entrenando Modelo v1_2 con ELO1 y Features Avanzadas ...")

history = modelo_elo1_v1_2.fit(
    data_entrada_elo1, data_salida_elo1, 
    epochs=2000, 
    verbose=0, 
    validation_split=0.1, 
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)]
)
print("modelo con ELO1 v1_2 entrenado")

# Imprimir la pérdida (Huber Loss en este caso) igual que en v1_1
loss_v1_2 = modelo_elo1_v1_2.evaluate(data_entrada_elo1_test, data_salida_elo1_test, verbose=0)[0]
print(f"Pérdida (Huber Loss) del modelo con ELO1_v1_2: {loss_v1_2:.4f}")

# Imprimir los resultados MAE y Precisión
acc_v1_2, mae_v1_2 = evaluar_precision(modelo_elo1_v1_2, data_entrada_elo1_test, data_salida_elo1_test, "Modelo ELO1_v1_2 Avanzado")


Entrenando Modelo v1_2 con ELO1 y Features Avanzadas ...
modelo con ELO1 v1_2 entrenado
Pérdida (Huber Loss) del modelo con ELO1_v1_2: 13.6623

--- Resultados Modelo ELO1_v1_2 Avanzado ---
Error Promedio Puntos (MAE Total): 9.83
  -> Error Medio Local: 9.78
  -> Error Medio Visitante: 9.87
Precisión Ganador (deducida): 64.63%


In [9]:
# ----------------- GUARDADO DE RESULTADOS -----------------
os.makedirs('resultados_finales', exist_ok=True)
def guardar_resultados_csv(df, modelo, entrada, nombre_archivo):
    df_copy = df.copy()
    df_copy = df_copy[['season_id', 'game_date', 'team_name_home', 'team_name_away', 'pts_home', 'pts_away']]
    predicciones = modelo.predict(entrada, verbose=0)
    df_copy['pred_pts_home'] = predicciones[:, 0].round(2)
    df_copy['pred_pts_away'] = predicciones[:, 1].round(2)
    df_copy['home_win'] = df_copy['pts_home'] > df_copy['pts_away']
    df_copy['pred_home_win'] = predicciones[:, 0] > predicciones[:, 1]
    df_copy['acierto'] = df_copy['home_win'] == df_copy['pred_home_win']
    df_copy.to_csv('resultados_finales/' + nombre_archivo, index=False)

guardar_resultados_csv(partidos_elo1_test, modelo_elo1_v1_2, data_entrada_elo1_test, 'resultados_modelo_v1_2.csv')
print("\nLos resultados detallados se han guardado en la carpeta 'resultados_finales'.")


Los resultados detallados se han guardado en la carpeta 'resultados_finales'.
